# Step 09 — Single Agent

🇬🇧 **English** (this notebook)

Move from a hand-written prompt to a CrewAI `Agent`. This notebook is **standalone** — the agent is defined right here in code, no separate YAML config or project files to edit. The `role`, `goal`, and `backstory` are still just a system prompt under the hood — CrewAI assembles it for you. What the framework adds is the loop: the agent reasons in steps before producing output, can call tools (Steps 10/11), and retries on failure.

## Learning objective

By the end of this notebook, you will:

- Understand what a CrewAI `Agent` is, and how it differs from the system-message-shaped prompts of Steps 04–08
- Understand the ReAct (Reason + Act) loop that lets an agent take multiple internal reasoning steps before producing a final answer
- Have run a standalone `Agent` — no `Task`, no `Crew` — on a real research question

## Prerequisites

- [Step 03 — Zero-Shot Prompting](step_03_zero_shot_prompting.ipynb) completed, with your team's topic chosen — this notebook reuses it
- The same `.env` setup as the previous steps
- If [Step 02 — Introduction to CrewAI](step_02_intro_to_crewai.ipynb) is new territory, at least skim it first — this notebook assumes you know what an `Agent`'s `role`/`goal`/`backstory` are

## Background

The core loop that makes an agent an agent — alternating between reasoning about what to do next and taking an action (calling a tool, reading a result, updating a plan) — was introduced in:

> Yao, S., et al. (2022). *ReAct: Synergizing Reasoning and Acting in Language Models*. ICLR 2023. [arXiv:2210.03629](https://arxiv.org/abs/2210.03629)

ReAct (Reason + Act) is the pattern CrewAI agents follow: the model thinks ("I need to find X"), acts, observes the result, thinks again, and repeats until it can produce a final answer. This loop is what separates an agent from a single prompt call.

## How this works

`Agent.kickoff(...)` runs a single agent standalone — no `Crew`, no `Task`, no YAML files needed. The cell below:

1. Defines the agent's identity as three plain strings — `role`, `goal`, `backstory` — the same three components `src/research_crew/config/agents.yaml` uses for this repo's full demo crew, just written directly here instead of loaded from a YAML file.
2. Creates the `Agent` and asks it a real research question with `agent.kickoff(question)`.
3. Uses `await` in front of `kickoff(...)` — Jupyter runs its own event loop internally, so `kickoff()` returns something that needs `await`ing inside a notebook specifically. A plain `.py` script wouldn't need this; it's purely a notebook quirk, not a CrewAI concept to learn.
4. Prints `result.raw` — the agent's final answer, after its internal reasoning is done.

The worked example runs a single **Researcher** agent investigating the EU AI Act. Steps 10–13 reuse this exact case, each adding one capability, so you can compare what each addition actually changes.

In [1]:
import os

from dotenv import load_dotenv
from crewai import Agent

load_dotenv()

# ── Agent identity — the same "researcher" template as
# src/research_crew/config/agents.yaml, with {topic} filled in via an
# f-string instead of CrewAI's own YAML interpolation ──────────────────────
topic = "EU AI Act"

role      = f"{topic} Senior Data Researcher"
goal      = f"Uncover cutting-edge developments in {topic}"
backstory = (
    f"You're a seasoned researcher with a knack for uncovering the latest "
    f"developments in {topic}. Known for your ability to find the most relevant "
    f"information and present it in a clear and concise manner."
)

agent = Agent(
    role=role,
    goal=goal,
    backstory=backstory,
    verbose=True,
)

# ── The question — same topic as previous steps ──────────────────────────────
question = (
    f"Explain {topic}'s risk-based categories (unacceptable, high-risk, "
    "limited, minimal) and what obligations apply to providers of high-risk AI systems."
)

# Jupyter runs its own event loop, and kickoff() detects that automatically and
# returns a coroutine instead of running synchronously - awaiting it is required here
# (this quirk only shows up in a notebook; a plain .py script would not need it).
result = await agent.kickoff(question)
print(result.raw)

╭───────────────────────────────────────────── 🤖 LiteAgent Started ──────────────────────────────────────────────╮
│                                                                                                                 │
│  LiteAgent Started                                                                                              │
│  Role: EU AI Act Senior Data Researcher                                                                         │
│  Status: In Progress                                                                                            │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── 🤖 LiteAgent Started ──────────────────────────────────────────────╮
│                                                                                                                 │
│  LiteAgent Session Started                                                                                      │
│  Name:                                                                                                          │
│  EU AI Act Senior Data Researcher                                                                               │
│  id:                                                                                                            │
│  8933585f-a0c0-47c5-ad43-3b112dd09177                                                                           │
│  role:                                                                                                          │
│  EU AI Act Senior Data Researcher                                                                               │
│  goal:                                                                                                          │
│  Uncover cutting-edge developments in EU AI Act                                                                 │
│  backstory:                                                                                                     │
│  You're a seasoned researcher with a knack for uncovering the latest developments in EU AI Act. Known for your  │
│  ability to find the most relevant information and present it in a clear and concise manner.                    │
│  tools:                                                                                                         │
│  []                                                                                                             │
│  verbose:                                                                                                       │
│  True                                                                                                           │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: EU AI Act Senior Data Researcher                                                                        │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  The EU AI Act employs a tiered, risk-based approach to regulation, ensuring that the stringency of             │
│  obligations is proportional to the potential impact of an AI system on fundamental rights and safety.          │
│                                                                                                                 │
│  Here is a breakdown of the four categories and the specific regulatory burden placed on "high-risk"            │
│  providers.                                                                                                     │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ### 1. The Risk-Based Hierarchy                                                                                │
│                                                                                                                 │
│  *   **Unacceptable Risk (Prohibited):** AI systems considered a clear threat to fundamental rights. These are  │
│  banned outright.                                                                                               │
│      *   *Examples:* Social scoring systems by governments, biometric categorization based on sensitive traits  │
│  (race, religion, etc.), and manipulative techniques that distort behavior to cause harm.                       │
│  *   **High-Risk:** AI systems permitted but subject to strict compliance obligations. These are systems used   │
│  in critical infrastructure, education, employment, essential private/public services, or law enforcement.      │
│  *   **Limited Risk:** AI systems with specific transparency requirements.                                      │
│      *   *Examples:* Chatbots (AI agents) or deepfakes. Users must be informed they are interacting with a      │
│  machine or viewing manipulated content.                                                                        │
│  *   **Minimal/No Risk:** The vast majority of AI applications (e.g., spam filters, AI-enabled video games).    │
│  These face no new regulatory obligations under the Act, though voluntary codes of conduct are encouraged.      │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ### 2. Obligations for High-Risk AI Providers                                                                  │
│                                                                                                                 │
│  If you are a provider of a high-risk AI system, the EU AI Act imposes a comprehensive lifecycle compliance     │
│  framework. You must implement the following:                                                                   │
│                                                                                                                 │
│  #### A. Risk Management System                        

The EU AI Act employs a tiered, risk-based approach to regulation, ensuring that the stringency of obligations is proportional to the potential impact of an AI system on fundamental rights and safety. 

Here is a breakdown of the four categories and the specific regulatory burden placed on "high-risk" providers.

---

### 1. The Risk-Based Hierarchy

*   **Unacceptable Risk (Prohibited):** AI systems considered a clear threat to fundamental rights. These are banned outright.
    *   *Examples:* Social scoring systems by governments, biometric categorization based on sensitive traits (race, religion, etc.), and manipulative techniques that distort behavior to cause harm.
*   **High-Risk:** AI systems permitted but subject to strict compliance obligations. These are systems used in critical infrastructure, education, employment, essential private/public services, or law enforcement.
*   **Limited Risk:** AI systems with specific transparency requirements.
    *   *Examples:* Chatbots (AI

╭──────────────────────────────────────────── ✅ LiteAgent Completed ─────────────────────────────────────────────╮
│                                                                                                                 │
│  LiteAgent Completed                                                                                            │
│  Role: EU AI Act Senior Data Researcher                                                                         │
│  id: 8933585f-a0c0-47c5-ad43-3b112dd09177                                                                       │
│  role: EU AI Act Senior Data Researcher                                                                         │
│  goal: Uncover cutting-edge developments in EU AI Act                                                           │
│  backstory: You're a seasoned researcher with a knack for uncovering the latest developments in EU AI Act.      │
│  Known for your ability to find the most relevant information and present it in a clear and concise manner.     │
│  tools: []                                                                                                      │
│  verbose: True                                                                                                  │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

## Your task

1. Run the cell. Read the verbose log above the final answer — this is the first time you can see the agent's internal reasoning, not just the final answer. Does the agent break the task into sub-steps?

2. Compare this Researcher's answer to the prompting steps' (Steps 04–08) plain-prompt answer on a similar question. What does having an explicit `role`/`goal`/`backstory` add, if anything, over a system message you wrote by hand — given that CrewAI compiles them into almost the same thing under the hood (see the Background note above)?

3. Now swap in your own team's topic: replace `role`, `goal`, `backstory`, and `question` with an agent suited to your topic. Keep that identity the same across Steps 10–13 so the comparisons stay meaningful.

4. Note what you observed — you'll draw on this for `REPORT.md`'s Section 3 (System Architecture) and 4.1 (LLM Selection & Configuration), once you start building your own agent. This closes out Sprint 3 — add its row to the **Sprint Progression** table before your interim submission.

## Shortcomings

One agent, however capable, can only reason over what it already knows — it has no way to check anything against the current state of the world, and it can't be simultaneously a credulous researcher gathering everything and a skeptical analyst questioning what it found.

[Step 10](step_10_tools.ipynb) addresses the first gap: giving the agent a tool it can call at runtime to ground its answer in current information. (The second gap — a second, complementary agent — gets its own step later, in [Step 13](step_13_multi_agent_seq.ipynb).)

## Resources for further reading

- Yao, S., et al. (2022). *ReAct: Synergizing Reasoning and Acting in Language Models*. ICLR 2023. [arXiv:2210.03629](https://arxiv.org/abs/2210.03629)
- Wang, L., et al. (2023). *A Survey on Large Language Model based Autonomous Agents*. [arXiv:2308.11432](https://arxiv.org/abs/2308.11432)
- [CrewAI `Agent` concept docs](https://docs.crewai.com/en/concepts/agents)

## Stretch goal

Look at the verbose log's "Final Answer" alongside the agent's intermediate reasoning. Find one place where the reasoning and the conclusion seem inconsistent. What does this tell you about trusting chain-of-thought?

---

**→ Interim submission is due after this step.** See [Assignment Overview](../../team_assignment/en/assignment-overview.md) for exactly what's expected.